# Day 46: The Machine-Learning Workflow

**Today's question:** how do we know whether a model has *really* learned something, and is not just
repeating answers it has already seen?

On Day 41 you "learned" a pass/fail rule by trying study-hour thresholds by hand. Today a library,
**scikit-learn**, does the learning, and we learn the honest way to check it.

Before running a cell marked 🤔, predict what it will show.

## 1. The supervised-learning workflow

Every project this week follows the same seven steps:

```
1. Question     "Will this student pass?"
2. Data         students.csv (298 students)
3. X and y      X = the clues (features), y = the answer (target)
4. Split        training set (to learn from) + test set (hidden until the end)
5. Baseline     the score of a "no-brain" guess; every model must beat it
6. Train        model.fit(X_train, y_train)
7. Evaluate     model.predict(X_test) and compare with y_test
   → Use        predict for a brand-new student
```

*Supervised* means every row in the training data comes with the right answer (`passed` is known).

## 2. Question, data, features `X` and target `y`

Question: **will a student pass?** The target is `passed` (1 = passed, 0 = failed).

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../../../datasets")
df = pd.read_csv(DATA_DIR / "students.csv")
df.head()

We start with two features. Notice the brackets:
- `df[["study_hours", "previous_score"]]` (**double** brackets, a list of names) gives a **table** → `X`
- `df["passed"]` (**single** brackets, one name) gives **one column** → `y`

In [ ]:
features = ["study_hours", "previous_score"]
X = df[features]
y = df["passed"]

print("X shape:", X.shape)
print("y shape:", y.shape)

`X` has 298 rows and 2 columns. `y` has 298 values: one answer per student.

## 3. Hide some students: the train/test split

**The exam analogy.** If a teacher grades students using the exact questions they practised, a student who
memorised the answers gets 100% without understanding anything. A fair exam uses **new** questions.

So we hide about 20% of the students in a **test set**. The model learns from the other 80% (the **training set**)
and is graded only on the hidden students.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("training rows:", len(X_train))
print("test rows:    ", len(X_test))

- `test_size=0.2` → 20% of rows go to the test set (60 of 298).
- The four results always come back in this order: **X before y, train before test**.
- The split is **random** (it shuffles first). `random_state=42` fixes the shuffle, so everyone in the room
  gets the same rows, today and next week. Any whole number works; 42 is just a habit.

🤔 **Predict:** if I run the split again with `random_state=42`, do I get the same first test students?
And with `random_state=7`?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test.head(3)

In [ ]:
X_train_7, X_test_7, y_train_7, y_test_7 = train_test_split(X, y, test_size=0.2, random_state=7)
X_test_7.head(3)

Same `random_state` → same split (students 159, 264, 254 first). Different `random_state` → different students.
The numbers on the left are the original row numbers, so you can see the rows were shuffled.

## 4. A baseline: the score to beat

Before any clever model, ask: *what score do I get with a no-brain guess?*
For a yes/no question the simplest guess is **"always say the most common answer"** from the training data.

In [ ]:
y_train.value_counts()

Most training students passed (173 of 238). So the no-brain rule is: **always predict 1 (pass)**.

🤔 **Predict:** about what accuracy will "always say pass" get on the test set?

In [ ]:
most_common = 1   # read from the counts above
correct = (y_test == most_common)
print(correct.head())
print("baseline accuracy:", correct.mean())

`y_test == 1` gives `True`/`False` for each test student. The **mean** of True/False values is the
**fraction that are True** (True counts as 1, False as 0). So the baseline is right for **65%** of the test
students without looking at any feature.

**Every model must beat this number.** A model with 66% accuracy is barely better than saying "pass" to everyone.

## 5. scikit-learn's pattern: create → `fit` → `predict`

scikit-learn has a ready-made "no-brain" model, `DummyClassifier`. It uses the same three steps as every real model.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")   # 1. create the model
baseline.fit(X_train, y_train)                          # 2. learn from the TRAINING data only
baseline_predictions = baseline.predict(X_test)         # 3. predict for the hidden TEST students

print(baseline_predictions[:10])

`.predict` returns one prediction per test row (a NumPy array). Accuracy = fraction of predictions that match
the real answers. `.score(X_test, y_test)` does that comparison for us.

In [ ]:
print("by hand:  ", (baseline_predictions == y_test).mean())
print("with score:", baseline.score(X_test, y_test))

## 6. A first real model: a tree of yes/no questions

A **decision tree** learns questions like *"is previous_score ≤ 64.5?"*. That is your Day 41 threshold search,
done automatically: it tries many thresholds on many features and keeps the one that separates pass from fail best.

`max_depth` = how many questions in a row the tree may ask.

🤔 **Predict:** a tree allowed only **one** question (`max_depth=1`). Will it beat the baseline of 0.65?

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

one_question = DecisionTreeClassifier(max_depth=1, random_state=42)
one_question.fit(X_train, y_train)
print("accuracy:", one_question.score(X_test, y_test))

Exactly the baseline! Let's look inside. `export_text` prints the questions the tree learned.

In [ ]:
print(export_text(one_question, feature_names=features))

The tree did learn a threshold (previous score 64.5), but **both** groups still mostly pass, so it says
"pass" either way. Learning *something* is not the same as being *useful*. Only the baseline comparison shows this.

Now allow two questions in a row:

In [ ]:
model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(X_train, y_train)
print("baseline accuracy:", baseline.score(X_test, y_test))
print("tree accuracy:    ", model.score(X_test, y_test))

In [ ]:
print(export_text(model, feature_names=features))

Read the first branch aloud: *"If the previous score is 64.5 or less **and** the student studies 7.65 hours or
less per week → predict fail."* The tree found these numbers itself from 238 training students.

0.733 beats 0.65: on these 60 hidden students the model gets 5 more students right than the baseline (44 vs 39).

## 7. Using the model: a brand-new student

A model is only useful if we can ask it about someone it has never seen. We give it a **table with the same
columns** as `X`. `pd.DataFrame([...])` builds a small table from a list of dictionaries, one dictionary per row.

In [ ]:
new_students = pd.DataFrame([
    {"study_hours": 12, "previous_score": 70},   # Asha
    {"study_hours": 4, "previous_score": 50},    # Omar
])
new_students

In [ ]:
model.predict(new_students)

Asha → 1 (pass), Omar → 0 (fail). The model gives an answer, not a guarantee. It is right about 73% of the
time on students like these.

## 8. Regression has a baseline too

Question 2: **what final score will a student get?** The target is a number, so the no-brain guess is
**"everyone gets the average training score"**. `DummyRegressor` does exactly that.

In [ ]:
from sklearn.dummy import DummyRegressor

y_score = df["final_score"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_score, test_size=0.2, random_state=42)

mean_model = DummyRegressor(strategy="mean")
mean_model.fit(Xr_train, yr_train)
mean_predictions = mean_model.predict(Xr_test)
print(mean_predictions[:5])

For numbers, "right or wrong" makes no sense. We ask: **how far off is each guess, on average?**
`.abs()` turns each difference into a distance (no minus signs).

In [ ]:
misses = (yr_test - mean_predictions).abs()
print("average miss:", round(misses.mean(), 1), "points")

Guessing the average misses by about **8.8 points** per student. Tomorrow we build a model that misses by less,
and give this number its official name.

## 9. Common mistakes (these cells fail **on purpose**)

In [ ]:
# Mistake 1: single brackets give ONE column (a Series), but X must be a table
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(df["study_hours"], y)

In [ ]:
# Mistake 2: predicting before fitting
fresh_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
fresh_tree.predict(X_test)

In [ ]:
# Mistake 3: the new data has different column names than the training data
model.predict(pd.DataFrame([{"hours": 12, "previous_score": 70}]))

**Fixes:** use `df[["study_hours"]]` (double brackets) · call `.fit` first · use exactly the training column names.

**Mistake 4 gives no error at all.** Scoring a model on the data it learned from:

In [ ]:
memoriser = DecisionTreeClassifier(random_state=42)   # no max_depth: as many questions as it likes
memoriser.fit(X_train, y_train)
print("score on TRAINING data:", memoriser.score(X_train, y_train))
print("score on TEST data:    ", memoriser.score(X_test, y_test))

Graded on the questions it practised, it looks brilliant. On new students it is worse than our two-question tree.
**Always report the test score.** (Day 49 is all about this gap.)

## Recap

1. Workflow: question → data → `X`/`y` → split → baseline → `fit` → evaluate on the test set → predict.
2. The test set is the hidden exam. The model never learns from it; we only grade with it.
3. `train_test_split(X, y, test_size=0.2, random_state=42)` returns `X_train, X_test, y_train, y_test`.
4. A baseline (most common class, or the mean) is the score to beat. Accuracy alone means nothing without it.
5. Every scikit-learn model: create → `.fit(X_train, y_train)` → `.predict(X_new)`.

**Next time:** regression. A line through the points, what its numbers mean, and measuring error in points.